In [1]:
from cgra import *
from kernels import *

In [2]:
kernel_name = "benchmarks/mmul/8x8"
version = "_gen"

In [3]:
# Global variables
CGRA_N_ROWS = 8
CGRA_N_COLS = 8
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &B[0][0]          &C[0][1]        &A[0][0]        nRowsBlocksC    -               &C[0][5]    -               -
    # nColsBlocksC      &B[0][1]        &C[1][2]        &A[1][0]        -               -           &C[1][6]        -
    # -                 loopColsA       &B[0][2]        &C[2][3]        &A[2][0]        -           -               &C[2][7]
    # &C[3][0]          -               nColsBlocksC    &B[0][3]        &C[3][4]        &A[3][0]    -               -
    # -                 &C[4][1]        -               -               &B[0][4]        &C[4][5]    &A[4][0]        -
    # -                 -               &C[5][2]        -               nColsBlocksC    &B[0][5]    &C[5][6]        &A[5][0]
    # &A[6][0]          -               -               &C[6][3]        -               -           &B[0][6]        &C[6][7]
    # &C[7][0]          &A[7][0]        -               -               &C[7][4]        -           nColsBlocksC    &B[0][7]
    # ----------------------
    # offset = 32*colsB - 32*nItL2
    # ----------------------
    # -4*colsB          colsA           -               -               -               offset          -               -
    # -                 -4*colsB        colsA           -               -               -               offset          -
    # -                 -               -4*colsB        colsA           -               -               -               offset
    # offset            -               -               -4*colsB        colsA           -               -               -
    # -                 offset          -               -               -4*colsB        colsA           -               -
    # -                 -               offset          -               -               -4*colsB        colsA           -
    # -                 -               -               offset          -               -               -4*colsB        colsA
    # colsA             -               -               -               offset          -               -               -4*colsB
    
    nItLoopColsA = colsA
    nColsBlocksC = int(colsB/CGRA_N_ROWS)
    nRowsBlocksC = int(rowsA/CGRA_N_ROWS)
    first_addr_A = first_addr
    #first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_B = 30000
    #first_addr_C = first_addr_B + colsA*colsB*4
    first_addr_C = 40000
    offset = 32*colsB - 32*nColsBlocksC

    config_vals_col = [[] for _ in range(CGRA_N_COLS)]

    config_vals_col[7] = [first_addr_C + 7*4 + 2*colsB*4,
                        first_addr_A + 5*colsA*4,
                        first_addr_C + 7*4 + 6*colsB*4,
                        first_addr_B + 7*4,
                        offset,
                        colsA,
                        -4*colsB]

    config_vals_col[6] = [first_addr_C + 6*4 + colsB*4,
                        first_addr_A + 4*colsA*4,
                        first_addr_C + 6*4 + 5*colsB*4,
                        first_addr_B + 6*4,
                        nColsBlocksC,
                        offset,
                        colsA,
                        -4*colsB]

    config_vals_col[5] = [first_addr_C + 5*4,
                        first_addr_A + 3*colsA*4,
                        first_addr_C + 5*4 + 4*colsB*4,
                        first_addr_B + 5*4,
                        offset,
                        colsA,
                        -4*colsB]

    config_vals_col[4] = [first_addr_A + 2*colsA*4,
                        first_addr_C + 4*4 + 3*colsB*4,
                        first_addr_B + 4*4,
                        nColsBlocksC,
                        first_addr_C + 7*colsB*4 + 4*4,
                        colsA,
                        -4*colsB,
                        offset]

    config_vals_col[3] = [nRowsBlocksC,
                        first_addr_A + colsA*4,
                        first_addr_C + 3*4 + 2*colsB*4,
                        first_addr_B + 3*4,
                        first_addr_C + 3*4 + 6*colsB*4,
                        colsA,
                        -4*colsB,
                        offset]

    config_vals_col[2] = [first_addr_A,
                        first_addr_C + colsB*4 + 2*4,
                        first_addr_B + 2*4,
                        nColsBlocksC,
                        first_addr_C + 5*colsB*4 + 2*4,
                        colsA,
                        -4*colsB,
                        offset]
    
    config_vals_col[1] = [ first_addr_C + 4,
                         first_addr_B + 4,
                         nItLoopColsA,
                         first_addr_C + 4*colsB*4 + 4,
                         first_addr_A + 7*colsA*4,
                         colsA,
                         -4*colsB,
                         offset]

    config_vals_col[0] = [first_addr_B,
                        nColsBlocksC,
                        first_addr_C + 3*colsB*4,
                        first_addr_A + 6*colsA*4,
                        first_addr_C + 7*colsB*4,
                        -4*colsB,
                        offset,
                        colsA]

    addr_config_loads = [0 for _ in range(CGRA_N_COLS)]
    for i in range(CGRA_N_COLS):
        kernel_add_memory_region(kernel_name, addr_config_loads[i], config_vals_col[i], version=version)
        if i < CGRA_N_COLS -1:
            addr_config_loads[i+1] = addr_config_loads[i] + len(config_vals_col[i])*4
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)

    return addr_config_loads

In [6]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [7]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def mmul_cpu(A_data, B_data, C_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum 
    return expected_res

In [9]:
# Test dimensions (5xXx5)
rowsA = 16
colsA = 9
colsB = 8
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
C_data = [x + 200 for x in range(0, rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB)

In [10]:
runKernel(load_addrs, max_it=20000)

Instr =  0 ( 0 )
[30000, 40004, 20000,    2,    0, 40020,    0,    0]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4, NOP , LWD R2  4, NOP , NOP ]    
[   1, 30004, 40040, 20036,    0,    0, 40056,    0]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4, NOP , NOP , LWD R2  4, NOP ]    
[   0,    9, 30008, 40076, 20072,    0,    0, 40092]    [NOP , LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4, NOP , NOP , LWD R2  4]    
[40096,    0,    1, 30012, 40112, 20108,    0,    0]    [LWD R2  4, NOP , LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4, NOP , NOP ]    
[   0, 40132,    0,    0, 30016, 40148, 20144,    0]    [NOP , LWD R2  4, NOP , NOP , LWD R2  4, LWD R2  4, LWD R2  4, NOP ]    
[   0,    0, 40168,    0,    1, 30020, 40184, 20180]    [NOP , NOP , LWD R2  4, NOP , LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20216,    0,    0, 40204,    0,    0, 30024, 40220]    [LWD R2  4, NOP , NOP , LWD R2  4, NOP , NOP , LWD R2  4, LWD R2  4]    
[40224, 20252,    0,    0, 40240,    0,    2, 30028]   

In [11]:
# Get result from CGRA
first_addr_C = 40000
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)
# Process estra rows/cols
if rowsA%5 != 0:
    for rA in range(rowsA - rowsA%4, rowsA):
        for cB in range(colsB):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum 
if colsB%5 != 0:
    for cB in range(colsB - colsB%4, colsB):
        for rA in range(rowsA - rowsA%4):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum

# Get cpu output
expected_res = mmul_cpu(A_data_cpy, B_data_cpy, C_data_cpy, rowsA, colsA, colsB)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")



Err: 26
Expected: 
[5232, 5268, 5304, 5340, 5376, 5412, 5448, 5484]
[15924, 16041, 16158, 16275, 16392, 16509, 16626, 16743]
[26616, 26814, 27012, 27210, 27408, 27606, 27804, 28002]
[37308, 37587, 37866, 38145, 38424, 38703, 38982, 39261]
[48000, 48360, 48720, 49080, 49440, 49800, 50160, 50520]
[58692, 59133, 59574, 60015, 60456, 60897, 61338, 61779]
[69384, 69906, 70428, 70950, 71472, 71994, 72516, 73038]
[80076, 80679, 81282, 81885, 82488, 83091, 83694, 84297]
[90768, 91452, 92136, 92820, 93504, 94188, 94872, 95556]
[101460, 102225, 102990, 103755, 104520, 105285, 106050, 106815]
[112152, 112998, 113844, 114690, 115536, 116382, 117228, 118074]
[122844, 123771, 124698, 125625, 126552, 127479, 128406, 129333]
[133536, 134544, 135552, 136560, 137568, 138576, 139584, 140592]
[144228, 145317, 146406, 147495, 148584, 149673, 150762, 151851]
[154920, 156090, 157260, 158430, 159600, 160770, 161940, 163110]
[165612, 166863, 168114, 169365, 170616, 171867, 173118, 174369]
CGRA: 
[5232, 5268, 5